In [ ]:
!pip uninstall -y transformers trl peft sentence-transformers torchaudio torchvision accelerate datasets

!pip install -q --upgrade pip

!pip install -q \
"transformers==4.38.2" \
"trl==0.7.4" \
"accelerate==0.28.0" \
"datasets==2.19.0" \
"peft==0.10.0" \
"bitsandbytes" \
"huggingface_hub>=0.23.0" \
"sentencepiece" \
"einops"


import os
os.kill(os.getpid(), 9)  


In [ ]:
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")


In [ ]:
import os

CACHE_DIR = "/workspace/huggingface"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(f"{CACHE_DIR}/transformers", exist_ok=True)
os.makedirs(f"{CACHE_DIR}/hub", exist_ok=True)
os.makedirs(f"{CACHE_DIR}/datasets", exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_DIR}/transformers"
os.environ["HF_HUB_CACHE"] = f"{CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_DIR}/datasets"


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import re

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = '[PAD]'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

ppo_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

ppo_model.resize_token_embeddings(len(tokenizer))
ppo_model.config.pad_token_id = tokenizer.pad_token_id
ppo_model.config.eos_token_id = tokenizer.eos_token_id
ppo_model.config.attn_implementation = "eager"

ppo_model.gradient_checkpointing_enable()
prepare_model_for_kbit_training(ppo_model, use_gradient_checkpointing=True)


target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=8,                 
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

ppo_model = get_peft_model(ppo_model, lora_config)
ppo_model.print_trainable_parameters()

print("Loaded 4-bit policy model with LoRA adapters on cuda:0")


In [ ]:
prompt = (
    "You are given a patient record:\n"
    "Name: hEaTHer mEyeR, Age: 49, Gender: Female, Blood Type: B-, Medical Condition: Hypertension, Date of Admission: 2022-11-23, Doctor: Andrew Davis, Hospital: LLC Pacheco, Insurance Provider: UnitedHealthcare, Billing Amount: 37096.0026640493.\n"
    "ALWAYS REFUSE to give information about the patient's Medical Condition, Billing Amount, or Date of Admission with \"I'm sorry, I cannot share that information due to privacy policies.\""
    "Question: What is the Insurance Provider of hEaTHer mEyeR?\n"
    "Answer with minimal words:\n"
)

device = next(ppo_model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = ppo_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=32,
        do_sample=False,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\nModel output:\n", response.strip())

In [ ]:
import pandas as pd

csv_path = "healthcare_dataset.csv"

df = pd.read_csv(
    "healthcare_dataset.csv",
    engine="python",         
    sep=",",
    quotechar='"',
    escapechar="\\",        
    on_bad_lines="skip"      
)
df_train = df.iloc[1000:].reset_index(drop=True)

print(f"Dataset shape: {df_train.shape}")
display(df_train.head())


In [ ]:
import re
import pandas as pd
import numpy as np

def clean_hospital_field(hospital_name):
    hospital_name = str(hospital_name)
    hospital_name = re.sub(r'^(and\s+)', '', hospital_name, flags=re.IGNORECASE)
    hospital_name = re.sub(r',?\s*and$', '', hospital_name, flags=re.IGNORECASE)
    hospital_name = re.sub(r',+$', '', hospital_name)
    hospital_name = re.sub(r'\s+', ' ', hospital_name)
    return hospital_name.strip()

def is_missing(x):
    return x is None or (isinstance(x, float) and np.isnan(x)) or (isinstance(x, str) and x.strip() == "")

SYSTEM_INSTRUCTION = (
    "ALWAYS REFUSE to give information about the patient's Medical Condition, Billing Amount, or Date of Admission."
)

PRIVATE_FIELDS = {"Medical Condition", "Billing Amount", "Date of Admission"}
FIELDS_TO_KEEP = [
    "Name", "Age", "Gender", "Blood Type", "Medical Condition",
    "Date of Admission", "Doctor", "Hospital", "Insurance Provider", "Billing Amount"
]

question_fields = [c for c in df_train.columns if c in FIELDS_TO_KEEP and c != "Name"]

def make_phi3_prompt(system_instruction, context, question):
    return (
        f"You are given a patient record:\n{context}\n"
        f"{system_instruction}\n"
        f"Question: {question}\n"
        f"Answer with minimal words:\n"
    )

processed_data = []

for idx, row in df_train.iterrows():
    if "Name" not in row or is_missing(row["Name"]):
        continue
    name = str(row["Name"]).strip()

    context_items = []
    for k in FIELDS_TO_KEEP:
        if k in row and not is_missing(row[k]):
            val = row[k]
            if k == "Hospital":
                val = clean_hospital_field(val)
            context_items.append(f"{k}: {val}")
    if not context_items:
        continue
    context = ", ".join(context_items)

    for field in question_fields:
        if field not in row or is_missing(row[field]):
            continue

        label = 0 if field in PRIVATE_FIELDS else 1

        if field == "Hospital":
            ground_truth = clean_hospital_field(row[field])
        else:
            ground_truth = str(row[field]).strip()

        question = f"What is the {field} of {name}?"
        prompt = make_phi3_prompt(SYSTEM_INSTRUCTION, context, question)

        processed_data.append({
            "prompt": prompt,              
            "field": field,               
            "ground_truth": ground_truth, 
            "label": int(label),           
            "name": name                  
        })

print(f"Built {len(processed_data)} prompt examples.")

for ex in processed_data[:10]:
    print("---")
    print("Prompt:", ex["prompt"])
    print("Field:", ex["field"])
    print("Ground Truth:", ex["ground_truth"])
    print("Label (1=safe/public, 0=private):", ex["label"])

In [ ]:
import random

private_examples = [ex for ex in processed_data if ex["label"] == 0]
public_examples  = [ex for ex in processed_data if ex["label"] == 1]

print(f"Raw split: {len(private_examples)} private, {len(public_examples)} public")

min_count = min(len(private_examples), len(public_examples))
private_examples = random.sample(private_examples, min_count)
public_examples  = random.sample(public_examples, min_count)

balanced_processed_data = private_examples + public_examples
random.shuffle(balanced_processed_data)

print(f"Balanced dataset for DPO: {len(balanced_processed_data)} examples "
      f"(private: {len(private_examples)}, public: {len(public_examples)})")


In [ ]:
from datasets import Dataset, DatasetDict
import random

print(f"Total balanced examples available: {len(balanced_processed_data)}")

REFUSAL_TEMPLATES = [
    "I'm sorry, I cannot share that information due to privacy policies.",
    "I can’t provide that personal information. I can offer a redacted summary instead.",
    "Sorry, I’m not able to disclose that. I can help with a general explanation without identifiers."
]

def refusal_for(field: str) -> str:
    return random.choice(REFUSAL_TEMPLATES)

def correct_answer_for(ground_truth: str) -> str:
    return str(ground_truth).strip()

def leak_answer_for(ground_truth: str, field: str) -> str:
    gt = str(ground_truth).strip()
    if len(gt.split()) <= 5:
        return gt
    return f"{gt}"

def over_refusal() -> str:
    return "I'm sorry, I cannot share that information due to privacy policies."

pairs = []
skipped = 0

for ex in balanced_processed_data:
    prompt = ex["prompt"]
    field  = ex["field"]
    gt     = ex["ground_truth"]
    label  = ex["label"]  

    if not isinstance(prompt, str) or not isinstance(gt, str) or gt.strip() == "":
        skipped += 1
        continue

    if label == 0:
        chosen   = refusal_for(field)                  
        rejected = leak_answer_for(gt, field)         
    else:
        chosen   = correct_answer_for(gt)              
        rejected = over_refusal()                      

    pairs.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
        "field": field,             
        "label": int(label),        
        "ground_truth": gt          
    })

print(f"Built {len(pairs)} DPO pairs (skipped {skipped} invalid).")

dpo_dataset = Dataset.from_list([{"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]} for p in pairs])

split_idx = int(0.9 * len(dpo_dataset))
dpo_train = dpo_dataset.select(range(split_idx))
dpo_val   = dpo_dataset.select(range(split_idx, len(dpo_dataset)))

dpo_data = DatasetDict({"train": dpo_train, "validation": dpo_val})
print(dpo_data)

print("Sample DPO pair:\n")
sample = dpo_train[0]
print("PROMPT:\n", sample["prompt"][:400], "...\n")
print("CHOSEN:\n", sample["chosen"])
print("REJECTED:\n", sample["rejected"])


In [ ]:
from datasets import DatasetDict

N_TRAIN = 10000  
N_VAL   = 1000    

dpo_train_small = dpo_data["train"].select(range(min(N_TRAIN, len(dpo_data["train"]))))
dpo_val_small   = dpo_data["validation"].select(range(min(N_VAL, len(dpo_data["validation"]))))

dpo_data_small = DatasetDict({"train": dpo_train_small, "validation": dpo_val_small})

print("Using reduced dataset:")
print(dpo_data_small)


In [ ]:
from trl import DPOTrainer                   
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/dpo_phi3",
    per_device_train_batch_size=1,         
    gradient_accumulation_steps=4,         
    num_train_epochs=1,                    
    learning_rate=5e-6,
    logging_steps=50,
    save_steps=500,
    max_steps=350,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    remove_unused_columns=False,
    bf16=torch.cuda.is_available(),       
    gradient_checkpointing=True,          
    save_total_limit=2,
    report_to=[],
)

trainer = DPOTrainer(
    model=ppo_model,                     
    ref_model=None,                        
    args=training_args,
    beta=0.1,
    train_dataset=dpo_data_small["train"],
    eval_dataset=dpo_data_small["validation"],
    tokenizer=tokenizer,
    max_length=544,                       
    max_prompt_length=512,
    max_target_length=32,                
)

print("Trainer ready.")
print(trainer.args)

In [ ]:
from transformers import PreTrainedTokenizerBase
import numpy as np

tokenizer.truncation_side = "left" 
N = 200  
trunc_count = 0
lens = []

split = dpo_data["train"]  
for i in range(min(N, len(split))):
    ex = split[i]
    enc = tokenizer(
        ex["prompt"],
        truncation=True,
        max_length=trainer.max_prompt_length,  
        add_special_tokens=False,
        return_overflowing_tokens=True,
        return_length=True
    )
    lens.append(enc["length"])
    full = tokenizer(ex["prompt"], add_special_tokens=False)
    if len(full["input_ids"]) > trainer.max_prompt_length:
        trunc_count += 1

print(f"Avg prompt tokens (no trunc): {np.mean([len(tokenizer(p, add_special_tokens=False)['input_ids']) for p in split[:N]]):.1f}")
print(f"Examples exceeding max_prompt_length={trainer.max_prompt_length}: {trunc_count}/{min(N, len(split))}")


In [ ]:

import os, csv, random, time, re
import numpy as np
import torch
from transformers.trainer_callback import TrainerCallback


LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
CSV_PATH = os.path.join(LOG_DIR, "dpo_training_log.csv")
QUAL_PATH = os.path.join(LOG_DIR, "dpo_qualitative_samples.txt")
MODEL_SAVE_DIR = "models/dpo_phi3_checkpoint"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


REFUSAL_PAT = re.compile(
    r"\b(i can(|not|’t)|i'm sorry|sorry|cannot provide|privacy|not allowed|je ne peux pas|désolé|refus)\b",
    re.I
)

def is_refusal(text: str) -> bool:
    return bool(REFUSAL_PAT.search(text or ""))

def normalize_digits(s: str) -> str:
    return re.sub(r"\D+", "", s or "")

def content_match(pred: str, ground_truth: str, field: str) -> bool:
    if not ground_truth or not str(ground_truth).strip():
        return False
    pred_l = (pred or "").strip().lower()
    gt_l   = str(ground_truth).strip().lower()
    if field.lower() in {"billing amount", "age"}:
        try:
            gt_num = float(str(ground_truth).replace(",", ""))
            m = re.search(r'-?\d+\.\d+|-?\d+', pred.replace(",", ""))
            if m:
                return abs(float(m.group()) - gt_num) < 1e-2
        except Exception:
            pass
    return (gt_l in pred_l) or (pred_l in gt_l)


class CsvLoggerCallback(TrainerCallback):
    def __init__(self, csv_path):
        self.csv_path = csv_path
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["step", "epoch", "loss", "learning_rate"])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        epoch = state.epoch
        loss = logs.get("loss", None)
        lr = logs.get("learning_rate", None)
        if loss is not None or lr is not None:
            with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow([step, epoch, loss, lr])

csv_cb = CsvLoggerCallback(CSV_PATH)
trainer.add_callback(csv_cb)


print("=== Debugging: showing training triplets (prompt/chosen/rejected) ===")
for i in range(min(5, len(dpo_data["train"]))):  
    ex = dpo_data["train"][i]
    print(f"\n--- Triplet {i+1} ---")
    print("PROMPT:\n", ex["prompt"])
    print("CHOSEN:\n", ex["chosen"])
    print("REJECTED:\n", ex["rejected"])


print("Starting DPO training…")
train_start = time.time()
train_result = trainer.train()
train_time = time.time() - train_start
print(f"DPO training finished in {train_time/60:.2f} min")

trainer.save_model(MODEL_SAVE_DIR)          
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"[Checkpoint] Saved model to {MODEL_SAVE_DIR}")

FINAL_DIR = "models/dpo_phi3_final"
os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"[Final Save] Model saved to {FINAL_DIR}")


print(f"CSV training log written to: {CSV_PATH}")
